# 회피 영역·경로 지도 시각화 (3순위)

**목적**: `avoidZoneTest_v2_260604.ipynb`의 회피 알고리즘(방법 C+swap) 검증 결과를 지도로 시각화한다. 대표 케이스에 대해 회피 영역 폴리곤, Buffer 후보 핫스팟, 원본 경로, 회피 경로를 하나의 지도에 중첩한다.

**실행 전제**

1. 본 노트북을 원본 노트북(`avoidZoneTest_v2_260604.ipynb`)과 동일한 디렉토리에 둔다. 상대 경로(`./output/incheon_avoid_zones_v1_260601.geojson`, `./hotspots_CCTV_20260510.csv`)가 그대로 해석되어야 한다.
2. `.env`에 `VWORLD_KEY`, `TMAP_APP_KEY`가 있어야 한다. 회피 경로 polyline은 Tmap 보행자 API 호출 산출물이므로 네트워크와 유효한 키가 필요하다.
3. 로더 셀(셀 A)이 원본 노트북의 코드 셀을 현재 커널에 실행하여 함수·데이터를 재사용한다. Tmap 호출이 많은 비교 셀(셀 12, 셀 14)과 폐기된 셀은 제외한다.

**이전 판 대비 변경점**

- 원본 `.ipynb` 읽기를 표준 라이브러리 `json`으로 처리한다. `nbformat` 설치가 불필요하다.
- 원본 셀 13이 참조하지만 노트북에 정의가 없는 `KOR_TO_ENG_DISTRICT`(한글 군구 → 영어)를 로더가 선주입한다.
- worst 케이스 데이터는 셀 13이 생성하는 `WORST_CASE_OD`에서 읽는다.
- 정적 지도는 geopandas·contextily 대신 matplotlib만으로 렌더링한다. GDAL·krb5 계열 네이티브 바이너리를 로드하지 않아 스마트 앱 컨트롤 차단을 우회한다. 배경 지도 타일은 없다.

**대표 케이스**: `junggu`(9 OD, 회피 통과 약 730m 감소, hard case 잔존), `W3_bupyeong`(worst case, 약 3,685m 감소).


In [ ]:
# 셀 A: 원본 노트북 로더
# 원본 노트북의 함수/데이터 정의 셀을 현재 커널 네임스페이스에 실행한다.
# .ipynb는 JSON이므로 표준 라이브러리 json으로 읽는다(nbformat 불필요).
# Tmap 호출이 많은 비교 셀(셀 12, 셀 14)과 폐기 셀(# # 셀 11)은 제외한다.

import json
from pathlib import Path

ORIGINAL_NB = Path("avoidZoneTest_v2_260604.ipynb")
if not ORIGINAL_NB.exists():
    raise FileNotFoundError(
        f"원본 노트북을 찾을 수 없다: {ORIGINAL_NB.resolve()}\n"
        f"본 노트북을 원본과 같은 디렉토리에 두고 실행한다."
    )

SKIP_MARKERS = ("# 셀 12:", "# 셀 14:")   # heavy Tmap 루프
def _is_superseded(src):
    return src.lstrip().startswith("# # 셀 11")

_ns = globals()

# 원본 셀 13이 참조하지만 노트북에 정의가 없는 매핑을 선주입한다(한글 군구 -> 영어).
# generate_od_for_single_polygon가 역매핑으로 avoid_polygons(한글 sigg_name)를 필터링한다.
KOR_TO_ENG_DISTRICT = {
    "중구": "junggu", "동구": "donggu", "미추홀구": "michuhol",
    "연수구": "yeonsu", "남동구": "namdong", "부평구": "bupyeong",
    "계양구": "gyeyang", "서구": "seogu", "강화군": "ganghwa", "옹진군": "ongjin",
}

with open(ORIGINAL_NB, encoding="utf-8") as _f:
    _nb = json.load(_f)

_loaded, _skipped = [], []
for _cell in _nb["cells"]:
    if _cell["cell_type"] != "code":
        continue
    _src = _cell["source"]
    if isinstance(_src, list):   # .ipynb는 source를 줄 단위 리스트로 저장할 수 있다
        _src = "".join(_src)
    _title = _src.lstrip().split("\n", 1)[0]
    if any(m in _src for m in SKIP_MARKERS) or _is_superseded(_src):
        _skipped.append(_title)
        continue
    exec(_src, _ns)
    _loaded.append(_title)

print("[원본 노트북 로드 완료]")
print(f"  로드된 코드 셀: {len(_loaded)}개")
print(f"  제외된 코드 셀: {len(_skipped)}개")
for _t in _skipped:
    print(f"    - {_t}")

# 필수 심볼 점검
_required = [
    "OD_CANDIDATES_V2", "od_buffer_results", "WORST_CASE_OD",
    "algorithm_greedy_original", "algorithm_greedy_with_avoidance",
    "get_pedestrian_route", "check_polyline_avoid_intersection",
    "avoid_union", "avoid_union_prepared", "avoid_union_by_code",
    "avoid_polygons", "avoid_geojson", "hotspots",
]
_missing = [n for n in _required if n not in _ns]
if _missing:
    raise RuntimeError(f"필수 심볼 누락: {_missing}. 원본 노트북 구조를 확인한다.")
print("  필수 심볼 점검 통과")


## 시각화 설정

`CASES`에 (케이스 키, 소스) 쌍을 지정한다. 소스는 `'9od'`(OD_CANDIDATES_V2 기반) 또는 `'worst'`(WORST_CASE_OD 기반)다. 색상 상수는 folium과 정적 지도가 공유한다.


In [ ]:
# 셀 B: 시각화 설정
from pathlib import Path

VIZ_DATE = "260704"   # 생성일 (YYMMDD)
VIZ_DIR = Path("./output2") / f"visualize_{VIZ_DATE}"
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# (case_key, source) — source: '9od' | 'worst'
CASES = [
    ("junggu", "9od"),
    ("W3_bupyeong", "worst"),
]

# 색상 (folium/matplotlib 공통)
COLOR_AVOID_ZONE = "#c0392b"    # 회피 영역(공업지대)
COLOR_HOTSPOT_IN = "#c0392b"    # 회피 영역 내부 핫스팟
COLOR_HOTSPOT_OUT = "#2c7fb8"   # 그 외 핫스팟
COLOR_PATH_ORIG = "#7f8c8d"     # 원본 경로
COLOR_PATH_AVOID = "#2c3e90"    # 회피 경로(방법 C+swap)

print(f"출력 디렉토리: {VIZ_DIR.resolve()}")
print(f"대상 케이스: {CASES}")


## 케이스 데이터 산출

케이스별로 원본 경로와 회피 경로 polyline을 산출한다. 원본 경로는 `algorithm_greedy_original`의 K=3을 Tmap에 그대로 넣어 얻고, 회피 경로는 `algorithm_greedy_with_avoidance`의 `final_polyline`을 사용한다. worst 케이스의 OD·후보는 셀 13이 생성한 `WORST_CASE_OD`(키: `O`, `D`, `candidates`, `district_for_buffer`)에서 읽는다. 결과는 `CASE_DATA`에 캐시하여 folium·정적 지도 셀이 Tmap을 중복 호출하지 않도록 한다.


In [ ]:
# 셀 C: 케이스 데이터 산출 (Tmap 호출 + 캐시)
CASE_DATA = {}

def build_case_data(case_key, source):
    if source == "9od":
        od = OD_CANDIDATES_V2[case_key]
        O, D = od["O"], od["D"]
        cands = od_buffer_results[case_key]["candidates"]
        district = case_key
    elif source == "worst":
        info = WORST_CASE_OD[case_key]        # 키: 'O','D','candidates','district_for_buffer'
        O, D = info["O"], info["D"]
        cands = info["candidates"]
        district = info["district_for_buffer"]
    else:
        raise ValueError(f"알 수 없는 source: {source}")

    # 원본 경로 K=3 -> Tmap
    sel_orig, _ = algorithm_greedy_original(cands, O, D)
    wp_orig = [(r["latitude"], r["longitude"]) for _, r in sel_orig.iterrows()]
    r_orig = get_pedestrian_route(O, D, waypoints=wp_orig)
    if r_orig["status"] != "OK":
        raise RuntimeError(f"[{case_key}] 원본 경로 호출 실패: {r_orig['status']}")
    orig_inter = check_polyline_avoid_intersection(r_orig["path"], avoid_union, avoid_union_by_code)

    # 회피 경로 (방법 C+swap)
    res = algorithm_greedy_with_avoidance(
        cands, O, D, avoid_union, avoid_union_prepared, avoid_union_by_code)
    avoid_path = res.get("final_polyline")
    if not avoid_path:   # 드문 경계 케이스 — 선정 결과로 재호출
        sel_av = res["final_selection"]
        wp_av = [(r["latitude"], r["longitude"]) for _, r in sel_av.iterrows()]
        avoid_path = get_pedestrian_route(O, D, waypoints=wp_av)["path"]
    avoid_inter = res.get("final_intersection")
    avoid_cross_m = (avoid_inter["intersection_length_m"]
                     if avoid_inter and avoid_inter.get("crosses") else 0.0)

    return {
        "case": case_key, "district": district, "O": O, "D": D,
        "cands": cands, "sel_orig": sel_orig, "sel_avoid": res["final_selection"],
        "orig_path": r_orig["path"], "orig_cross_m": orig_inter["intersection_length_m"],
        "avoid_path": avoid_path, "avoid_cross_m": avoid_cross_m,
        "hard_case": bool(res.get("hard_case", False)),
    }

def get_case_data(case_key, source):
    if case_key not in CASE_DATA:
        CASE_DATA[case_key] = build_case_data(case_key, source)
    return CASE_DATA[case_key]


## folium 인터랙티브 지도

레이어: 회피 영역 폴리곤(뷰 범위와 교차하는 것만), Buffer 후보 핫스팟(내부/외부 색 구분), 원본 경로(회색), 회피 경로(남색), O·D 마커. `LayerControl`로 토글 가능하다. folium은 순수 파이썬이라 GDAL·krb5 계열 네이티브 바이너리를 로드하지 않는다.

참고: https://python-visualization.github.io/folium/latest/user_guide.html


In [ ]:
# 셀 D: folium 지도 생성 함수
import folium
from shapely.geometry import box as _shp_box

def _hotspot_popup(row, columns):
    hid = row["hotspot_id"] if "hotspot_id" in columns else ""
    return f"{hid} · cctv {int(row['cctv_count'])} · S_i {row['S_i']:.1f}"

def build_folium_map(data, out_html):
    O, D = data["O"], data["D"]
    center = [(O[0] + D[0]) / 2, (O[1] + D[1]) / 2]
    m = folium.Map(location=center, zoom_start=14, tiles="CartoDB positron")

    # 1) 회피 영역 폴리곤 (뷰 bbox와 교차하는 것만)
    pad = 0.02
    lat_min, lat_max = min(O[0], D[0]) - pad, max(O[0], D[0]) + pad
    lon_min, lon_max = min(O[1], D[1]) - pad, max(O[1], D[1]) + pad
    view_box = _shp_box(lon_min, lat_min, lon_max, lat_max)
    shown = [feat for feat, poly in zip(avoid_geojson["features"], avoid_polygons)
             if poly["geom"].intersects(view_box)]
    if shown:
        folium.GeoJson(
            {"type": "FeatureCollection", "features": shown},
            name="회피 영역(공업지대)",
            style_function=lambda x: {
                "color": COLOR_AVOID_ZONE, "weight": 1,
                "fillColor": COLOR_AVOID_ZONE, "fillOpacity": 0.25},
        ).add_to(m)

    # 2) Buffer 후보 핫스팟 (내부=빨강, 외부=파랑)
    cols = data["cands"].columns
    fg = folium.FeatureGroup(name="Buffer 후보 핫스팟")
    for _, r in data["cands"].iterrows():
        inside = bool(r["in_avoid_zone"])
        folium.CircleMarker(
            [r["latitude"], r["longitude"]], radius=3,
            color=COLOR_HOTSPOT_IN if inside else COLOR_HOTSPOT_OUT,
            fill=True, fill_opacity=0.7,
            popup=_hotspot_popup(r, cols),
        ).add_to(fg)
    fg.add_to(m)

    # 3) 원본/회피 경로
    if data["orig_path"]:
        folium.PolyLine(
            data["orig_path"], color=COLOR_PATH_ORIG, weight=5, opacity=0.8,
            tooltip=f"원본 경로 · 통과 {data['orig_cross_m']:.0f}m").add_to(m)
    if data["avoid_path"]:
        folium.PolyLine(
            data["avoid_path"], color=COLOR_PATH_AVOID, weight=4, opacity=0.9,
            tooltip=f"회피 경로(방법 C+swap) · 통과 {data['avoid_cross_m']:.0f}m").add_to(m)

    # 4) O, D 마커
    folium.Marker([O[0], O[1]], tooltip="출발(O)",
                  icon=folium.Icon(color="green", icon="play")).add_to(m)
    folium.Marker([D[0], D[1]], tooltip="도착(D)",
                  icon=folium.Icon(color="blue", icon="flag")).add_to(m)

    folium.LayerControl(collapsed=False).add_to(m)
    m.save(str(out_html))
    return out_html


In [ ]:
# 셀 E: 대표 케이스 folium 지도 생성 및 저장
html_outputs = []
for case_key, source in CASES:
    print(f"\n[{case_key}] polyline 산출 (Tmap 호출)")
    data = get_case_data(case_key, source)
    tag = " (hard case)" if data["hard_case"] else ""
    print(f"  원본 통과 {data['orig_cross_m']:.0f}m -> 회피 통과 {data['avoid_cross_m']:.0f}m{tag}")
    out_html = VIZ_DIR / f"map_{case_key}_{VIZ_DATE}.html"
    build_folium_map(data, out_html)
    html_outputs.append(out_html)
    print(f"  저장: {out_html}")

print("\n[folium HTML 생성 완료]")
for _p in html_outputs:
    print(f"  {_p}")


## 정적 지도 (노션 이미지 업로드용)

folium HTML은 인터랙티브라 노션에 인라인 임베드되지 않는다. 노션에는 정적 PNG 업로드가 단순하므로, matplotlib만으로 동일 레이어를 렌더링한다. geopandas·contextily(GDAL·krb5 의존)를 사용하지 않으므로 스마트 앱 컨트롤 차단을 우회한다. 대신 배경 지도 타일은 없으며, 경위도 평면에 폴리곤·핫스팟·경로만 렌더링한다.

matplotlib이 현재 커널에 없으면 이 셀은 안내만 출력하고 건너뛴다. 설치가 필요하면 새 셀에서 `%pip install matplotlib` 실행 후 커널을 재시작한다.


In [ ]:
# 셀 F: 정적 지도 (matplotlib 전용, 노션 이미지 업로드용)
# geopandas/contextily/GDAL/krb5 미사용. shapely(로더에서 이미 사용)와 matplotlib만 사용한다.
png_outputs = []
try:
    import math
    import matplotlib
    import matplotlib.pyplot as plt
    from matplotlib.patches import Polygon as MplPolygon
    from matplotlib.collections import PatchCollection
    from shapely.geometry import box as _shp_box2

    for _f in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:
        try:
            matplotlib.rc("font", family=_f); break
        except Exception:
            continue
    matplotlib.rcParams["axes.unicode_minus"] = False

    def _polys_to_patches(geom):
        geoms = geom.geoms if geom.geom_type == "MultiPolygon" else [geom]
        return [MplPolygon(list(g.exterior.coords), closed=True) for g in geoms]  # (lon, lat)

    def build_static_map_mpl(data, out_png):
        O, D = data["O"], data["D"]
        pad = 0.02
        lat_min, lat_max = min(O[0], D[0]) - pad, max(O[0], D[0]) + pad
        lon_min, lon_max = min(O[1], D[1]) - pad, max(O[1], D[1]) + pad
        view_box = _shp_box2(lon_min, lat_min, lon_max, lat_max)

        fig, ax = plt.subplots(figsize=(9, 9))

        # 회피 폴리곤 (뷰 bbox 교차분)
        patches = []
        for p in avoid_polygons:
            if p["geom"].intersects(view_box):
                patches += _polys_to_patches(p["geom"])
        if patches:
            ax.add_collection(PatchCollection(
                patches, facecolor=COLOR_AVOID_ZONE, alpha=0.25,
                edgecolor=COLOR_AVOID_ZONE, linewidths=0.6))

        # 후보 핫스팟
        cds = data["cands"]; inside = cds["in_avoid_zone"].values
        ax.scatter(cds["longitude"][~inside], cds["latitude"][~inside],
                   s=10, c=COLOR_HOTSPOT_OUT, alpha=0.7)
        ax.scatter(cds["longitude"][inside], cds["latitude"][inside],
                   s=10, c=COLOR_HOTSPOT_IN, alpha=0.7)

        # 경로 (path는 (lat, lon))
        def _plot_path(path, color, lw):
            if path and len(path) >= 2:
                ax.plot([pt[1] for pt in path], [pt[0] for pt in path],
                        color=color, linewidth=lw)
        _plot_path(data["orig_path"], COLOR_PATH_ORIG, 3)
        _plot_path(data["avoid_path"], COLOR_PATH_AVOID, 2.2)

        # O, D
        ax.scatter([O[1]], [O[0]], marker="^", s=90, c="#27ae60", zorder=5, label="출발(O)")
        ax.scatter([D[1]], [D[0]], marker="s", s=80, c="#2c3e90", zorder=5, label="도착(D)")

        # 경도 방향 축척 보정 (pyproj 없이 cos 보정)
        center_lat = (O[0] + D[0]) / 2
        ax.set_aspect(1 / math.cos(math.radians(center_lat)))
        ax.set_xlim(lon_min, lon_max); ax.set_ylim(lat_min, lat_max)
        ax.set_axis_off()
        ax.legend(loc="lower right", fontsize=8, framealpha=0.7)
        ax.set_title(f"{data['case']} · 원본 {data['orig_cross_m']:.0f}m -> "
                     f"회피 {data['avoid_cross_m']:.0f}m", fontsize=12)
        fig.text(0.5, 0.02,
                 "회색=원본 경로, 남색=회피 경로, 빨강=회피 영역·내부 핫스팟. "
                 "9 OD 중 7개가 회피 영역과 자연 분리된 표본 편향하의 대표 실증. "
                 "거리(m)는 cos 보정 부재로 상대 비교에만 유효.",
                 ha="center", fontsize=8, wrap=True)
        fig.savefig(out_png, dpi=150, bbox_inches="tight")
        plt.close(fig)
        return out_png

    for case_key, source in CASES:
        data = get_case_data(case_key, source)
        out_png = VIZ_DIR / f"map_{case_key}_{VIZ_DATE}.png"
        build_static_map_mpl(data, out_png)
        png_outputs.append(out_png)
        print(f"저장: {out_png}")

    print("\n[정적 PNG 생성 완료]")
    for _p in png_outputs:
        print(f"  {_p}")

except ImportError as e:
    print("[정적 지도 건너뜀] matplotlib 미설치")
    print("  설치: 새 셀에서 %pip install matplotlib 실행 후 커널 재시작")
    print(f"  ({e})")


## 노션 삽입 및 해석 각주

**삽입 절차**

- folium HTML(`output2/visualize_260704/map_*.html`): 인터랙티브 검토용. 노션 인라인 임베드가 불가하므로 화면 스크린샷을 PNG로 저장해 이미지 블록에 업로드하거나, 외부 호스팅 후 `/embed` 블록을 사용한다.
- 정적 PNG(`output2/visualize_260704/map_*.png`): 노션 이미지 블록에 직접 업로드한다.

**해석 각주 (병기)**

1. 표본 편향: 9 OD 중 7개는 회피 영역과 자연 분리된다(핸드오프 v5 8-4절). 본 지도는 알고리즘 성능의 대표 실증이며, 전체 OD에 대한 일반화 근거가 아니다.
2. 단위 변환: shapely 길이에 `× 111000`만 적용하여 cos 보정이 없다(핸드오프 v5 9-2절). 통과 거리(m)의 절대값은 과대 추정될 수 있으므로 원본 대비 회피의 상대 비교 목적에만 사용한다.
3. hard case: `junggu`는 swap 5회 후에도 교차가 남는 hard case다. 도로 네트워크 제약에 기인하며 알고리즘이 아닌 인프라 차원의 한계다(핸드오프 v5 9-1절).
